# 🌀 Attracteur de Clifford (Pickover) — Exploration ultra-complète

Ce notebook est un **laboratoire visuel interactif** dédié à l'étude de l'attracteur de Clifford,
une famille de systèmes dynamiques discrets introduite par **Clifford Pickover** dans les années 1990,
popularisée ensuite par Paul Bourke.

## Définition mathématique

L'attracteur est défini par la récurrence :

$$
\begin{cases}
x_{n+1} = \sin(a\,y_n) + c\cos(a\,x_n) \\
y_{n+1} = \sin(b\,x_n) + d\cos(b\,y_n)
\end{cases}
$$

où $(a, b, c, d)$ sont quatre paramètres réels (généralement pris dans $[-3, 3]$) qui pilotent
totalement la morphologie de l'attracteur : filaments, nappes, spirales, structures fractales...

Le point remarquable est que ce système, bien que **totalement déterministe**, produit des trajectoires
d'une richesse visuelle proche du chaos, tout en restant borné dans le plan (d'où le terme *attracteur*).

## Ce que propose ce notebook

1. Génération rapide (option **Numba JIT**) des trajectoires
2. Rendu statique haute qualité (scatter + densité)
3. **Exploration interactive** avec sliders (ipywidgets) — tous les paramètres sont "bidouillables"
4. Une **galerie** de configurations célèbres
5. Un rendu **Plotly WebGL** pour un zoom/pan fluide, à la souris
6. Une **marche aléatoire** dans l'espace des paramètres (générateur de variations)
7. Une **animation** d'interpolation entre deux attracteurs
8. Un système de **sauvegarde de favoris** (JSON)
9. Un bonus : comparaison avec les attracteurs cousins (de Jong, Svensson, Bedhead)

> 💡 Astuce : dans JupyterLab, exécutez `Kernel > Restart & Run All` pour tout initialiser proprement,
> puis jouez avec les sliders de la section 3.


## 0. Imports & configuration

On tente d'importer **Numba** pour compiler la boucle de génération en JIT (gain de x20 à x50 sur
de gros nombres de points). Si Numba n'est pas disponible, on retombe automatiquement sur une
implémentation NumPy pure, un peu plus lente mais 100% fonctionnelle.

On tente également d'importer **ipywidgets** et **plotly**, indispensables pour les sections
interactives. Le notebook reste utilisable en mode "statique" si l'un de ces paquets manque
(des messages d'avertissement s'afficheront).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from functools import lru_cache
import json, os, time, io

plt.rcParams['figure.facecolor'] = '#0d1117'

# --- Numba (accélération JIT) ---------------------------------------------
try:
    from numba import njit
    NUMBA_OK = True
except ImportError:
    NUMBA_OK = False
    def njit(*args, **kwargs):
        # décorateur factice si numba absent
        def wrap(f):
            return f
        if len(args) == 1 and callable(args[0]):
            return args[0]
        return wrap

# --- ipywidgets --------------------------------------------------------
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    WIDGETS_OK = True
except ImportError:
    WIDGETS_OK = False

# --- plotly --------------------------------------------------------------
try:
    import plotly.graph_objects as go
    PLOTLY_OK = True
except ImportError:
    PLOTLY_OK = False

print(f"Numba disponible      : {NUMBA_OK}")
print(f"ipywidgets disponible : {WIDGETS_OK}")
print(f"plotly disponible     : {PLOTLY_OK}")
if not NUMBA_OK:
    print(">> conseil : pip install numba  (accélère fortement la génération)")
if not WIDGETS_OK:
    print(">> conseil : pip install ipywidgets  (nécessaire pour les sliders interactifs)")
if not PLOTLY_OK:
    print(">> conseil : pip install plotly  (nécessaire pour le rendu zoom/pan fluide)")


## 1. Génération des trajectoires

Deux implémentations :

- `clifford_numpy` : pure NumPy, portable partout, correcte pour quelques centaines de milliers de points.
- `clifford_numba` : boucle compilée JIT, utilisable jusqu'à plusieurs dizaines de millions de points
  en un temps raisonnable.

Un cache (`lru_cache`) mémorise les derniers calculs pour que le déplacement des sliders sur des
paramètres déjà visités soit **instantané**.


In [ ]:
def clifford_numpy(a, b, c, d, n=200_000, x0=0.1, y0=0.0, skip=0):
    """Génère n points de l'attracteur de Clifford (implémentation NumPy pure).
    skip : nombre d'itérations de mise en régime (warm-up) à ignorer avant l'enregistrement.
    """
    x, y = x0, y0
    for _ in range(skip):
        x, y = np.sin(a*y) + c*np.cos(a*x), np.sin(b*x) + d*np.cos(b*y)
    xs, ys = np.empty(n), np.empty(n)
    for i in range(n):
        x, y = np.sin(a*y) + c*np.cos(a*x), np.sin(b*x) + d*np.cos(b*y)
        xs[i], ys[i] = x, y
    return xs, ys


@njit(fastmath=True)
def _clifford_core(a, b, c, d, n, x0, y0, skip):
    x, y = x0, y0
    for _ in range(skip):
        x, y = np.sin(a*y) + c*np.cos(a*x), np.sin(b*x) + d*np.cos(b*y)
    xs = np.empty(n, dtype=np.float64)
    ys = np.empty(n, dtype=np.float64)
    for i in range(n):
        x, y = np.sin(a*y) + c*np.cos(a*x), np.sin(b*x) + d*np.cos(b*y)
        xs[i] = x
        ys[i] = y
    return xs, ys


def clifford_numba(a, b, c, d, n=200_000, x0=0.1, y0=0.0, skip=0):
    return _clifford_core(float(a), float(b), float(c), float(d), int(n), float(x0), float(y0), int(skip))


# Choix automatique de la meilleure implémentation disponible
clifford_fast = clifford_numba if NUMBA_OK else clifford_numpy


@lru_cache(maxsize=64)
def clifford_cached(a, b, c, d, n=200_000, x0=0.1, y0=0.0, skip=0):
    """Version mise en cache (les arguments doivent être hashables, donc des floats/int simples)."""
    xs, ys = clifford_fast(a, b, c, d, n, x0, y0, skip)
    return xs, ys


# --- petit benchmark indicatif ---
t0 = time.time()
xs, ys = clifford_fast(-1.4, 1.6, 1.0, 0.7, n=300_000)
t1 = time.time()
print(f"Génération de 300 000 points en {t1 - t0:.3f} s "
      f"({'Numba JIT' if NUMBA_OK else 'NumPy pur'})")


## 2. Rendu statique — scatter & densité

Deux modes de rendu sont proposés :

- **`mode='scatter'`** : chaque point est dessiné, coloré selon son indice temporel (dégradé le long
  de la trajectoire) ou selon une colormap fixe. Rendu organique, fidèle au "grain" du nuage de points.
- **`mode='density'`** : on accumule les points dans une grille 2D (`np.histogram2d`), puis on affiche
  le logarithme de la densité. Rendu plus "lisse", façon carte de chaleur, qui met en valeur les zones
  de forte fréquentation de l'attracteur.


In [ ]:
def plot_clifford_static(a, b, c, d, n=200_000, x0=0.1, y0=0.0, skip=0,
                          mode='scatter', cmap='plasma', point_size=0.15, alpha=0.6,
                          bg_color='#0d1117', bins=900, figsize=(8, 8), ax=None,
                          title=True, gamma=0.4):
    """Affiche l'attracteur de Clifford pour un jeu de paramètres donné."""
    xs, ys = clifford_cached(round(a, 6), round(b, 6), round(c, 6), round(d, 6),
                              int(n), round(x0, 6), round(y0, 6), int(skip))

    created_fig = ax is None
    if created_fig:
        fig, ax = plt.subplots(figsize=figsize, facecolor=bg_color)
    ax.set_facecolor(bg_color)
    ax.clear()
    ax.set_facecolor(bg_color)

    if mode == 'scatter':
        colors = np.linspace(0, 1, len(xs))
        ax.scatter(xs, ys, s=point_size, c=colors, cmap=cmap, alpha=alpha, linewidths=0)
    else:  # density
        h, xedges, yedges = np.histogram2d(xs, ys, bins=bins)
        h = np.power(h, gamma)  # compression gamma pour révéler les faibles densités
        ax.imshow(h.T, origin='lower', extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
                  cmap=cmap, interpolation='bilinear')

    ax.set_xlim(xs.min() - 0.1, xs.max() + 0.1)
    ax.set_ylim(ys.min() - 0.1, ys.max() + 0.1)
    ax.set_axis_off()
    ax.set_aspect('equal')
    if title:
        ax.set_title(f"a={a:.3f}  b={b:.3f}  c={c:.3f}  d={d:.3f}   (n={n:,})",
                      color='white', fontsize=10, family='monospace')
    if created_fig:
        plt.tight_layout()
        plt.show()
    return ax


# Rendu du jeu de paramètres d'origine
plot_clifford_static(-1.4, 1.6, 1.0, 0.7, n=300_000, mode='scatter', cmap='plasma')


In [ ]:
# Le même attracteur, rendu en mode densité (souvent plus spectaculaire)
plot_clifford_static(-1.4, 1.6, 1.0, 0.7, n=1_500_000, mode='density', cmap='inferno', bins=1000, gamma=0.35)


## 3. 🎛️ Exploration interactive — le "bidouillage" en direct

C'est le cœur de ce notebook : **tous** les paramètres de l'attracteur sont exposés via des sliders
`ipywidgets`. Le rendu se met à jour en direct pendant que vous déplacez les curseurs
(`continuous_update=False` pour rester fluide même sur les gros nombres de points — le rendu se
déclenche au relâchement du slider).

Paramètres disponibles :

| Widget | Rôle |
|---|---|
| `a, b, c, d` | les 4 paramètres de la récurrence de Clifford |
| `x0, y0` | point de départ de la trajectoire |
| `n_points` | nombre de points générés (log-slider, 10k → 3M) |
| `skip` | nombre d'itérations de warm-up ignorées (élimine le "transitoire") |
| `mode` | scatter / density |
| `colormap` | palette matplotlib |
| `point_size`, `alpha` | rendu du mode scatter |
| `bg_color` | couleur de fond |
| Bouton **🎲 Aléatoire** | tire un nouveau jeu de paramètres au hasard |
| Bouton **♻️ Reset** | revient au jeu de paramètres d'origine |


In [ ]:
if WIDGETS_OK:
    style = {'description_width': '90px'}
    layout_slider = widgets.Layout(width='420px')

    a_s = widgets.FloatSlider(value=-1.4, min=-3, max=3, step=0.01, description='a', style=style, layout=layout_slider, continuous_update=False)
    b_s = widgets.FloatSlider(value=1.6, min=-3, max=3, step=0.01, description='b', style=style, layout=layout_slider, continuous_update=False)
    c_s = widgets.FloatSlider(value=1.0, min=-3, max=3, step=0.01, description='c', style=style, layout=layout_slider, continuous_update=False)
    d_s = widgets.FloatSlider(value=0.7, min=-3, max=3, step=0.01, description='d', style=style, layout=layout_slider, continuous_update=False)

    x0_s = widgets.FloatSlider(value=0.1, min=-2, max=2, step=0.01, description='x0', style=style, layout=layout_slider, continuous_update=False)
    y0_s = widgets.FloatSlider(value=0.0, min=-2, max=2, step=0.01, description='y0', style=style, layout=layout_slider, continuous_update=False)

    n_s = widgets.SelectionSlider(
        options=[10_000, 30_000, 100_000, 300_000, 1_000_000, 3_000_000],
        value=300_000, description='n_points', style=style, layout=layout_slider)
    skip_s = widgets.IntSlider(value=0, min=0, max=5000, step=100, description='skip', style=style, layout=layout_slider)

    mode_s = widgets.ToggleButtons(options=['scatter', 'density'], value='scatter', description='mode', style=style)
    cmap_s = widgets.Dropdown(
        options=['plasma', 'inferno', 'magma', 'viridis', 'cividis', 'twilight',
                 'turbo', 'cool', 'spring', 'hot', 'cubehelix', 'gist_heat'],
        value='plasma', description='colormap', style=style)
    bg_s = widgets.ColorPicker(value='#0d1117', description='bg_color', style=style)

    size_s = widgets.FloatSlider(value=0.15, min=0.02, max=2.0, step=0.02, description='point_size', style=style, layout=layout_slider, continuous_update=False)
    alpha_s = widgets.FloatSlider(value=0.6, min=0.05, max=1.0, step=0.05, description='alpha', style=style, layout=layout_slider, continuous_update=False)
    gamma_s = widgets.FloatSlider(value=0.4, min=0.1, max=1.0, step=0.05, description='gamma (density)', style=style, layout=layout_slider, continuous_update=False)

    random_btn = widgets.Button(description='🎲 Aléatoire', button_style='warning')
    reset_btn = widgets.Button(description='♻️ Reset', button_style='info')
    save_btn = widgets.Button(description='💾 Sauver ce favori', button_style='success')
    out = widgets.Output()

    DEFAULTS = dict(a=-1.4, b=1.6, c=1.0, d=0.7, x0=0.1, y0=0.0)

    def redraw(*_):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(7, 7), facecolor=bg_s.value)
            plot_clifford_static(
                a_s.value, b_s.value, c_s.value, d_s.value,
                n=n_s.value, x0=x0_s.value, y0=y0_s.value, skip=skip_s.value,
                mode=mode_s.value, cmap=cmap_s.value,
                point_size=size_s.value, alpha=alpha_s.value,
                bg_color=bg_s.value, gamma=gamma_s.value, ax=ax)
            plt.show()

    def on_random(_):
        a_s.value = round(np.random.uniform(-3, 3), 3)
        b_s.value = round(np.random.uniform(-3, 3), 3)
        c_s.value = round(np.random.uniform(-3, 3), 3)
        d_s.value = round(np.random.uniform(-3, 3), 3)
        redraw()

    def on_reset(_):
        a_s.value, b_s.value = DEFAULTS['a'], DEFAULTS['b']
        c_s.value, d_s.value = DEFAULTS['c'], DEFAULTS['d']
        x0_s.value, y0_s.value = DEFAULTS['x0'], DEFAULTS['y0']
        redraw()

    FAVORITES_PATH = 'clifford_favorites.json'

    def on_save(_):
        favs = []
        if os.path.exists(FAVORITES_PATH):
            with open(FAVORITES_PATH) as f:
                favs = json.load(f)
        favs.append(dict(a=a_s.value, b=b_s.value, c=c_s.value, d=d_s.value,
                          x0=x0_s.value, y0=y0_s.value, cmap=cmap_s.value,
                          mode=mode_s.value, bg=bg_s.value, ts=time.time()))
        with open(FAVORITES_PATH, 'w') as f:
            json.dump(favs, f, indent=2)
        with out:
            print(f"✅ Favori sauvegardé ({len(favs)} au total) dans {FAVORITES_PATH}")

    random_btn.on_click(on_random)
    reset_btn.on_click(on_reset)
    save_btn.on_click(on_save)

    for w in [a_s, b_s, c_s, d_s, x0_s, y0_s, n_s, skip_s, mode_s, cmap_s, bg_s, size_s, alpha_s, gamma_s]:
        w.observe(redraw, names='value')

    controls = widgets.VBox([
        widgets.HTML("<b>Paramètres de la récurrence</b>"),
        widgets.HBox([a_s, b_s]), widgets.HBox([c_s, d_s]),
        widgets.HTML("<b>Point de départ</b>"),
        widgets.HBox([x0_s, y0_s]),
        widgets.HTML("<b>Rendu</b>"),
        widgets.HBox([n_s, skip_s]),
        mode_s,
        widgets.HBox([cmap_s, bg_s]),
        widgets.HBox([size_s, alpha_s, gamma_s]),
        widgets.HBox([random_btn, reset_btn, save_btn]),
    ])

    display(widgets.HBox([controls, out]))
    redraw()
else:
    print("ipywidgets n'est pas installé : section interactive indisponible. "
          "Exécutez `pip install ipywidgets` puis redémarrez le kernel.")


## 4. 🖼️ Galerie de configurations célèbres

Quelques jeux de paramètres $(a, b, c, d)$ réputés pour produire des morphologies remarquables
(spirales, nappes, toiles, structures en "ailes"...). N'hésitez pas à en recopier les valeurs dans
les sliders de la section précédente pour les affiner.


In [ ]:
GALLERY = [
    dict(name="Original Pickover", a=-1.4, b=1.6, c=1.0, d=0.7),
    dict(name="Toile d'araignée", a=-1.7, b=1.3, c=-0.1, d=-1.2),
    dict(name="Ailes de papillon", a=1.7, b=1.7, c=0.06, d=1.2),
    dict(name="Nappes croisées", a=-1.7, b=1.8, c=-0.9, d=-0.4),
    dict(name="Symétrie fine", a=1.1, b=-1.32, c=-1.03, d=1.54),
    dict(name="Filaments denses", a=1.5, b=-1.8, c=1.6, d=0.9),
    dict(name="Structure en spirale", a=-1.9, b=-1.9, c=-1.9, d=-1.9),
    dict(name="Dispersion large", a=-2.0, b=-2.0, c=-1.2, d=2.0),
    dict(name="Nuage organique", a=1.4, b=-1.6, c=1.0, d=0.7),
]

n_cols = 3
n_rows = int(np.ceil(len(GALLERY) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows), facecolor='#0d1117')
axes = np.array(axes).reshape(-1)

for ax, cfg in zip(axes, GALLERY):
    plot_clifford_static(cfg['a'], cfg['b'], cfg['c'], cfg['d'], n=150_000,
                          mode='scatter', cmap='plasma', point_size=0.1, alpha=0.5,
                          ax=ax, title=False)
    ax.set_title(f"{cfg['name']}\n({cfg['a']}, {cfg['b']}, {cfg['c']}, {cfg['d']})",
                  color='white', fontsize=9)

for ax in axes[len(GALLERY):]:
    ax.axis('off')

plt.tight_layout()
plt.show()


## 5. 🔍 Rendu Plotly — zoom & pan natifs, très fluides

Le rendu Matplotlib est excellent pour l'export, mais son zoom interactif reste limité en Jupyter.
`plotly` avec `Scattergl` (rendu WebGL, accéléré GPU) permet un zoom/pan **très fluide** même avec
plusieurs centaines de milliers de points, en gardant une interactivité proche du temps réel.

Molette = zoom, glisser = pan, double-clic = reset.


In [ ]:
def plot_clifford_plotly(a, b, c, d, n=400_000, x0=0.1, y0=0.0, skip=0,
                          cmap='Plasma', point_size=1.4, bg_color='#0d1117'):
    if not PLOTLY_OK:
        print("plotly n'est pas installé : `pip install plotly`")
        return None
    xs, ys = clifford_cached(round(a, 6), round(b, 6), round(c, 6), round(d, 6),
                              int(n), round(x0, 6), round(y0, 6), int(skip))
    colors = np.linspace(0, 1, len(xs))

    fig = go.Figure(data=go.Scattergl(
        x=xs, y=ys, mode='markers',
        marker=dict(size=point_size, color=colors, colorscale=cmap, opacity=0.65, line_width=0),
    ))
    fig.update_layout(
        template='plotly_dark',
        paper_bgcolor=bg_color, plot_bgcolor=bg_color,
        width=800, height=800,
        title=f"Attracteur de Clifford — a={a}, b={b}, c={c}, d={d} (zoom/pan à la souris)",
        xaxis=dict(visible=False, scaleanchor='y'),
        yaxis=dict(visible=False),
        margin=dict(l=10, r=10, t=50, b=10),
    )
    fig.show()
    return fig

plot_clifford_plotly(-1.4, 1.6, 1.0, 0.7, n=400_000, cmap='Plasma')


## 6. 🚶 Marche aléatoire dans l'espace des paramètres

Plutôt que de tirer $(a,b,c,d)$ complètement au hasard (ce qui donne souvent des nuages informes ou
des trajectoires divergentes/dégénérées), on effectue une **marche aléatoire locale** autour d'un
point de départ intéressant : à chaque étape, on ajoute une petite perturbation gaussienne aux 4
paramètres. Cela permet de "dériver" continûment d'une forme à une autre voisine, en restant dans
des zones visuellement riches.


In [ ]:
def random_walk_gallery(start=(-1.4, 1.6, 1.0, 0.7), steps=6, step_size=0.25, seed=42, n=150_000):
    rng = np.random.default_rng(seed)
    params = [np.array(start, dtype=float)]
    for _ in range(steps - 1):
        nxt = params[-1] + rng.normal(scale=step_size, size=4)
        nxt = np.clip(nxt, -3, 3)
        params.append(nxt)

    n_cols = 3
    n_rows = int(np.ceil(steps / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows), facecolor='#0d1117')
    axes = np.array(axes).reshape(-1)
    for ax, p in zip(axes, params):
        plot_clifford_static(*p, n=n, mode='scatter', cmap='inferno',
                              point_size=0.1, alpha=0.5, ax=ax, title=False)
        ax.set_title(f"({p[0]:.2f}, {p[1]:.2f}, {p[2]:.2f}, {p[3]:.2f})", color='white', fontsize=8)
    for ax in axes[steps:]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    return params

_ = random_walk_gallery()


## 7. 🎞️ Animation — interpolation entre deux attracteurs

On interpole linéairement les 4 paramètres entre un jeu de départ et un jeu d'arrivée, et on génère
une image par étape : le résultat est une **morphose fluide** d'un attracteur vers un autre,
exportée en GIF (via Pillow, aucune dépendance externe type ffmpeg nécessaire).

⚠️ Cellule volontairement désactivée par défaut (`RUN_ANIMATION = False`) car le rendu de plusieurs
dizaines de frames peut prendre 1 à 2 minutes. Passez la variable à `True` pour la lancer.


In [ ]:
import matplotlib.animation as animation

RUN_ANIMATION = False  # passez à True pour générer le GIF

def make_morph_gif(start, end, n_frames=40, n_points=80_000, out_path='clifford_morph.gif',
                    cmap='plasma', fps=12):
    start, end = np.array(start, dtype=float), np.array(end, dtype=float)
    fig, ax = plt.subplots(figsize=(6, 6), facecolor='#0d1117')

    def update(frame):
        t = frame / (n_frames - 1)
        a, b, c, d = start + t * (end - start)
        plot_clifford_static(a, b, c, d, n=n_points, mode='scatter',
                              cmap=cmap, point_size=0.2, alpha=0.6, ax=ax, title=False)
        return []

    anim = animation.FuncAnimation(fig, update, frames=n_frames, blit=False)
    anim.save(out_path, writer='pillow', fps=fps)
    plt.close(fig)
    print(f"✅ Animation sauvegardée : {out_path}")
    return out_path

if RUN_ANIMATION:
    make_morph_gif(start=(-1.4, 1.6, 1.0, 0.7), end=(1.7, 1.7, 0.06, 1.2), n_frames=40)
else:
    print("RUN_ANIMATION = False -> cellule ignorée (passez la variable à True pour générer le GIF).")


## 8. 💾 Favoris — sauvegarde & rechargement

Le bouton **💾 Sauver ce favori** de la section interactive écrit les paramètres courants dans
`clifford_favorites.json`. Les cellules ci-dessous permettent de lister ces favoris et de recharger
n'importe lequel d'entre eux directement dans un rendu.


In [ ]:
FAVORITES_PATH = 'clifford_favorites.json'

def list_favorites():
    if not os.path.exists(FAVORITES_PATH):
        print("Aucun favori enregistré pour le moment (utilisez le bouton 💾 dans la section 3).")
        return []
    with open(FAVORITES_PATH) as f:
        favs = json.load(f)
    for i, fdict in enumerate(favs):
        print(f"[{i}] a={fdict['a']:.3f} b={fdict['b']:.3f} c={fdict['c']:.3f} d={fdict['d']:.3f} "
              f"cmap={fdict.get('cmap', '-')} mode={fdict.get('mode', '-')}")
    return favs

def render_favorite(index, n=300_000):
    favs = list_favorites()
    if not favs or index >= len(favs):
        print("Index invalide.")
        return
    f = favs[index]
    plot_clifford_static(f['a'], f['b'], f['c'], f['d'], n=n,
                          x0=f.get('x0', 0.1), y0=f.get('y0', 0.0),
                          mode=f.get('mode', 'scatter'), cmap=f.get('cmap', 'plasma'),
                          bg_color=f.get('bg', '#0d1117'))

favs = list_favorites()
# Exemple : render_favorite(0)


## 9. 🖨️ Export haute résolution

Fonction utilitaire pour exporter un rendu en très haute résolution (impression, fond d'écran,
usage artistique), avec un grand nombre de points et une taille de figure élevée.


In [ ]:
def export_hd(a, b, c, d, filename='clifford_hd.png', n=3_000_000, mode='density',
              cmap='inferno', bins=1600, dpi=200, figsize=(10, 10), bg_color='#0d1117'):
    fig, ax = plt.subplots(figsize=figsize, facecolor=bg_color, dpi=dpi)
    plot_clifford_static(a, b, c, d, n=n, mode=mode, cmap=cmap, bins=bins,
                          bg_color=bg_color, ax=ax, title=False)
    plt.savefig(filename, dpi=dpi, facecolor=bg_color, bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    print(f"✅ Export haute résolution enregistré : {filename}")

# Exemple (décommentez pour lancer un export ~quelques dizaines de secondes) :
# export_hd(-1.4, 1.6, 1.0, 0.7, filename='clifford_hd.png', n=3_000_000)


## 10. 🎁 Bonus — la famille des attracteurs de Pickover

L'attracteur de Clifford appartient à une famille plus large d'attracteurs 2D "map-based" popularisés
par Clifford Pickover et Paul Bourke. En voici deux cousins, pour comparaison :

- **Peter de Jong** : $x_{n+1} = \sin(a y_n) - \cos(b x_n)$, $\; y_{n+1} = \sin(c x_n) - \cos(d y_n)$
- **Svensson** : $x_{n+1} = d\sin(a x_n) - \sin(b y_n)$, $\; y_{n+1} = c\cos(a x_n) + \cos(b y_n)$

Ces variantes utilisent la même logique de récurrence trigonométrique à 4 paramètres, mais avec une
combinaison sin/cos différente, ce qui change radicalement la morphologie obtenue.


In [ ]:
@njit(fastmath=True)
def _dejong_core(a, b, c, d, n, x0, y0):
    x, y = x0, y0
    xs = np.empty(n); ys = np.empty(n)
    for i in range(n):
        x, y = np.sin(a*y) - np.cos(b*x), np.sin(c*x) - np.cos(d*y)
        xs[i], ys[i] = x, y
    return xs, ys

@njit(fastmath=True)
def _svensson_core(a, b, c, d, n, x0, y0):
    x, y = x0, y0
    xs = np.empty(n); ys = np.empty(n)
    for i in range(n):
        x, y = d*np.sin(a*x) - np.sin(b*y), c*np.cos(a*x) + np.cos(b*y)
        xs[i], ys[i] = x, y
    return xs, ys

def plot_family(core_fn, a, b, c, d, n=300_000, cmap='viridis', ax=None, title=''):
    xs, ys = core_fn(float(a), float(b), float(c), float(d), int(n), 0.1, 0.0)
    created = ax is None
    if created:
        fig, ax = plt.subplots(figsize=(6, 6), facecolor='#0d1117')
    ax.set_facecolor('#0d1117')
    colors = np.linspace(0, 1, len(xs))
    ax.scatter(xs, ys, s=0.15, c=colors, cmap=cmap, alpha=0.6, linewidths=0)
    ax.set_axis_off(); ax.set_aspect('equal')
    ax.set_title(title, color='white', fontsize=10)
    if created:
        plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 6), facecolor='#0d1117')
plot_family(_dejong_core, 1.4, -2.3, 2.4, -2.1, cmap='cool', ax=axes[0], title='Peter de Jong')
plot_family(_svensson_core, 1.4, -2.3, 2.4, -2.1, cmap='spring', ax=axes[1], title='Svensson')
plt.tight_layout()
plt.show()


## 🔚 Conclusion & pistes pour aller plus loin

- Essayez des valeurs extrêmes de $a, b, c, d$ proches de $\pm 3$ : certaines zones font diverger ou
  dégénérer l'attracteur en un simple point périodique — une exploration intéressante en soi.
- Combinez le mode `density` avec un `gamma` très bas (0.1–0.2) pour révéler des structures fractales
  très fines, invisibles en mode scatter.
- Le fichier `clifford_favorites.json` généré par ce notebook peut servir de base à une **galerie
  permanente** de vos plus belles trouvailles.
- Pour aller plus loin : attracteurs de **Bedhead**, **Gumowski-Mira**, ou l'extension de Clifford en
  3D (ajout d'une troisième récurrence en $z$).

Bonne exploration ! 🌌
